# NASA Mission Intelligence: end-to-end test run

**Author:** Dennis O'Higgins

This notebook runs the complete workflow on Google Colab: it installs the project, builds the ChromaDB collection with the embedding pipeline, inspects the collection, runs the batch RAGAS evaluation on `evaluation_dataset.txt`, and tests the Streamlit chat application with real questions.

The OpenAI API key is read from Colab Secrets (`OPENAI_API_KEY`, and `OPENAI_BASE_URL` for a Udacity/Vocareum key) and is never printed or saved.

## 1. Install the project and the NASA documents

In [ ]:
!rm -rf /content/nasa /content/udc
!git clone -q https://github.com/dohigg1987/nasa-mission-rag-chat.git /content/nasa
!git clone -q --depth 1 https://github.com/udacity/cd13318-exercises-project.git /content/udc
%cd /content/nasa
!cp -r "/content/udc/Project-NASA-Mission-Intelligence-Starter/data_text" .
!pip -q install -r requirements.txt
!python fix_ragas_imports.py
!python -W ignore -c "import ragas, chromadb, openai, streamlit; print('ragas', ragas.__version__, '| chromadb', chromadb.__version__, '| openai', openai.__version__, '| streamlit', streamlit.__version__)"
!ls data_text/*

## 2. Load the OpenAI key from Colab Secrets

In [ ]:
import os
from google.colab import userdata

key = None
for name in ("OPENAI_API_KEY", "OPENAI_API_Key", "OPENAI_KEY"):  # accept common spellings of the secret name
    try:
        key = userdata.get(name)
        break
    except Exception:
        continue
os.environ["OPENAI_API_KEY"] = key or ""
try:
    base_url = userdata.get("OPENAI_BASE_URL")
    if base_url:
        os.environ["OPENAI_BASE_URL"] = base_url
except Exception:
    pass  # a standard OpenAI key needs no base URL
os.environ["CHROMA_OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("Custom base URL set:", bool(os.environ.get("OPENAI_BASE_URL")))

## 3. Build the ChromaDB collection (chunk size 500, overlap 100)

In [ ]:
!python embedding_pipeline.py --data-path ./data_text --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text --chunk-size 500 --chunk-overlap 100 \
    --batch-size 100 --update-mode skip --test-query "What problem did Apollo 13 encounter?" 2>&1 \
    | grep -vE "Processing |chunks: " 

## 4. Inspect the collection (`--stats-only`)

In [ ]:
!python embedding_pipeline.py --stats-only --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text 2>&1 | sed 's/^.* - INFO - //'

## 5. Re-run in `skip` mode: existing chunks are detected and not re-embedded

In [ ]:
!python embedding_pipeline.py --data-path ./data_text --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text --chunk-size 500 --chunk-overlap 100 \
    --update-mode skip 2>&1 | grep -E "Documents added|Documents updated|Documents skipped|Total documents" | sed 's/^.* - INFO - //'

## 6. Retrieval check: top-k with and without a mission filter

In [ ]:
import rag_client
collection, ok, error = rag_client.initialize_rag_system("./chroma_db_openai", "nasa_space_missions_text")
print("Collection ready:", ok, error or "", "| chunks:", collection.count())
for mission in ["all", "apollo_13"]:
    res = rag_client.retrieve_documents(collection, "What problem did Apollo 13 encounter?", n_results=4, mission_filter=mission)
    print(f"\nmission_filter={mission}: {len(res['documents'][0])} results")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  {dist:.3f}  {meta['mission']:<10} {meta['source'][:45]:<45} chunk {meta['chunk_index']}")
print()
print(rag_client.format_context(res["documents"][0][:2], res["metadatas"][0][:2])[:1500])

## 7. Batch evaluation on `evaluation_dataset.txt`

In [ ]:
!python -W ignore batch_evaluation.py --dataset evaluation_dataset.txt --k 5 2>&1 | grep -vE "Evaluating|it/s\]" 

## 8. Streamlit chat application test (real questions, real scores)

In [ ]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file("chat.py", default_timeout=300).run()
print("Sidebar controls:", [s.label for s in at.selectbox], [s.label for s in at.slider])
at.selectbox[2].set_value("apollo_13").run()   # mission focus

for question in ["Who were the Apollo 13 crew members?",
                 "What did they do after the explosion?"]:  # follow-up uses conversation history
    at.chat_input[0].set_value(question).run()
    print("\nQ:", question)
    print("A:", [m.markdown[0].value for m in at.chat_message if m.markdown][-1])
    print("Scores:", [(m.label, m.value) for m in at.metric])
print("\nExceptions:", list(at.exception))

## 9. Package the results

In [ ]:
!zip -q -j /content/nasa_results.zip evaluation_results.json evaluation_results.csv chroma_embedding_text_only.log
!ls -la /content/nasa_results.zip evaluation_results.*